In [1]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType

In [2]:
pyspark_home = os.path.dirname(pyspark.__file__)

os.environ['SPARK_HOME'] = pyspark_home
os.environ['PATH'] += os.pathsep + os.path.join(pyspark_home, 'bin')
# Manually point to your Java and Hadoop locations
os.environ['JAVA_HOME'] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ['HADOOP_HOME'] = r"C:\hadoop"

# Force the 'bin' folders into the session path
os.environ['PATH'] += os.pathsep + os.path.join(os.environ['JAVA_HOME'], 'bin')
os.environ['PATH'] += os.pathsep + os.path.join(os.environ['HADOOP_HOME'], 'bin')

# Tell Spark exactly which python to use
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


if "spark" in globals():
    spark.stop()
spark = SparkSession.builder.appName("CarCleaning").config("spark.sql.ansi.enabled", "false").getOrCreate()
sc = spark.sparkContext

In [3]:
df = spark.read.csv("data/raw/craigslistVehicles.csv", header=True, inferSchema=False, multiLine=True, escape='"')

# Print the top 10 rows
df.show(5)
size_in_bytes = df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()

cleaning_logs = []
beforedata = {
            'total_rows': df.count(),
            'total_columns': len(df.columns),
            'memory_usage': f"{size_in_bytes / 1024**2:.2f} MB",
            'scan_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

+--------------------+----------------+--------------------+-----+----+------------+--------------------+---------+-----------+------+--------+------------+------------+-----------------+-----+--------+-----+-----------+--------------------+--------------------+---------+----------+
|                 url|            city|            city_url|price|year|manufacturer|                make|condition|  cylinders|  fuel|odometer|title_status|transmission|              VIN|drive|    size| type|paint_color|           image_url|                desc|      lat|      long|
+--------------------+----------------+--------------------+-----+----+------------+--------------------+---------+-----------+------+--------+------------+------------+-----------------+-----+--------+-----+-----------+--------------------+--------------------+---------+----------+
|https://grandrapi...|grand rapids, MI|https://grandrapi...| 1500|2006|    cadillac|                 cts|     good|6 cylinders|   gas|  236000|     

In [4]:
def add_to_report(check_type, column_affected, log):
    """Internal helper to standardize report entries."""
    report = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'check_type': check_type,
        'column_affected': column_affected,
        'log': log
    }
    cleaning_logs.append(report)

In [5]:
def clean_inconsistencies(df):
    total_rows = df.count()
    logs = []
    # --- Clean Cylinders ---
    # regexp_extract(column, regex, group) 
    # '\d+' looks for one or more digits. If it doesn't find any (like in 'other'), it returns an empty string.
    print("\nCleaning 'cylinders' column...")
    
    df_cleaned = df.withColumn(
        "cylinders_numeric", 
        F.regexp_extract(F.col("cylinders"), r"(\d+)", 1).cast("int")
    )

    # --- 3. Calculate New Null Percentage for Cylinders ---
    # Since "other" doesn't have digits, cast("int") turns it into NULL automatically.
    stats = df_cleaned.select(
        F.count(F.when(F.col("cylinders_numeric").isNull(), 1)).alias("null_count")
    ).collect()[0]

    new_null_count = stats["null_count"]
    new_null_pct = (new_null_count / total_rows) * 100

    print(f"⚙️ Cylinders (Numeric) Missing Percentage: {new_null_pct:.2f}%")
    print(f"   (Includes original nulls and rows that were 'other')")

    # Show the transformation result
    print("\nSample of transformed cylinders:")
    df_cleaned.select("cylinders", "cylinders_numeric").distinct().show()
    logs.append("Cylinders column cleaned of inconsistencies.")
    add_to_report("Inconsistency Cleaning","cylinders", logs)
    return df_cleaned

df1 = clean_inconsistencies(df)


Cleaning 'cylinders' column...
⚙️ Cylinders (Numeric) Missing Percentage: 40.30%
   (Includes original nulls and rows that were 'other')

Sample of transformed cylinders:
+------------+-----------------+
|   cylinders|cylinders_numeric|
+------------+-----------------+
| 4 cylinders|                4|
|10 cylinders|               10|
| 3 cylinders|                3|
| 8 cylinders|                8|
|12 cylinders|               12|
| 5 cylinders|                5|
| 6 cylinders|                6|
|        NULL|             NULL|
|       other|             NULL|
+------------+-----------------+



In [6]:
# 1. Define your Expected Schema

schema = StructType([
    StructField("url", StringType(), True),
    StructField("city", StringType(), True),
    StructField("city_url", StringType(), True),
    StructField("price", LongType(), True),
    StructField("year", IntegerType(), True),
    StructField("manufacturer", StringType(), True),
    StructField("make", StringType(), True),
    StructField("condition", StringType(), True),
    StructField("cylinders", StringType(), True),
    StructField("fuel", StringType(), True),
    StructField("odometer", DoubleType(), True),
    StructField("title_status", StringType(), True),
    StructField("transmission", StringType(), True),
    StructField("VIN", StringType(), True),
    StructField("drive", StringType(), True),
    StructField("size", StringType(), True),
    StructField("type", StringType(), True),
    StructField("paint_color", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("desc", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("cylinders_numeric", IntegerType(), True)
])


df_casted = df1
for field in schema.fields:
    col_name = field.name
    target_type = field.dataType
    # Apply the cast from the schema blueprint to the column
    df_casted = df_casted.withColumn(col_name, F.col(col_name).cast(target_type))
logs=["Columns casted to expected types."]
add_to_report("Data Type Validation","All", logs)
# Now df_casted has the actual Longs, Integers, and Doubles.
# Let's verify:
df_casted.printSchema()

root
 |-- url: string (nullable = true)
 |-- city: string (nullable = true)
 |-- city_url: string (nullable = true)
 |-- price: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- make: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- cylinders_numeric: integer (nullable = true)



In [7]:
def cap_outliers(df, numeric_cols):
    """
    Caps numerical columns at the Upper Bound (Q3 + 1.5*IQR)
    and saves a histogram of the result.
    """
    for col_name in numeric_cols:
        print(f"\nProcessing {col_name}...")
        
        # 1. Get the Upper Bound
        quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.01)
        if len(quantiles) < 2: continue
        
        q1, q3 = quantiles[0], quantiles[1]
        iqr = q3 - q1
        upper_bound = q3 + 1.5 * iqr
        lower_bound = q1 - 1.5 * iqr
        
        # 2. CAP THE DATA
        # If value > upper_bound, set it TO upper_bound. Else leave it.
        # We also ensure we handle NULLs by using otherwise()
        
        lower_outliers = df.filter(
            (F.col(col_name) < lower_bound)).count()
        if lower_outliers > 0:
            df = df.withColumn(col_name, 
                            F.when(F.col(col_name) < lower_bound, lower_bound)
                                .otherwise(F.col(col_name)))
            
            logs = [f"Capped {col_name} at lower bound: {lower_bound:.2f}"]
            add_to_report("Outlier Capping", col_name, logs)

        upper_outliers = df.filter(
            (F.col(col_name) > upper_bound)).count()
        if upper_outliers > 0:
            df = df.withColumn(col_name, 
                               F.when(F.col(col_name) > upper_bound, upper_bound)
                                .otherwise(F.col(col_name)))

            logs = [f"Capped {col_name} at upper bound: {upper_bound:.2f}"]
            add_to_report("Outlier Capping", col_name, logs)
            
    return df

columns_to_cap=["price","odometer", "lat", "long"]
df_capped = cap_outliers(df_casted, columns_to_cap)


Processing price...

Processing odometer...

Processing lat...

Processing long...


In [8]:
from pyparsing import col


def remove_nulls(df, columns):
    for col in columns:
        # 1. Check if column exists to avoid the typo trap
        if col not in df.columns:
            print(f"⚠️ Warning: Column '{col}' not found in DataFrame. Skipping.")
            continue
            
        dtype = dict(df.dtypes)[col]
        
        # 1. Base condition: check for actual NULLs
        condition = F.col(col).isNull()
        
        # 2. Add string-specific checks ONLY if it's a string column
        if dtype == "string":
            condition = condition | \
                        (F.col(col) == "") | \
                        (F.col(col) == "NULL") | \
                        (F.lower(F.col(col)) == "other")
        
        # Count only for reporting
        null_count = df.filter(condition).count()
        if null_count > 0:
            removed_df = df.filter(condition)
            removed_df.toPandas().to_csv(f"data/interim/removed_nulls_{col}.csv", index=False)
            df = df.filter(~condition)
            logs = [f"Removed {null_count} rows with NULLs in {col}"]
            add_to_report("Null Removal", col, logs)

    return df

columns_to_remove_nulls=["year", "title_status", "transmission", "lat", "long"]
df_no_nulls = remove_nulls(df_capped, columns_to_remove_nulls)

In [9]:
def fill_with_unknown(df, columns):
    """
    Replaces NULL, empty strings, literal "NULL", and "other" 
    with the string "Unknown".
    """
    for col in columns:
        # 1. Define what we consider "dirty"
        is_dirty_cond = (F.col(col).isNull()) | \
                        (F.col(col) == "") | \
                        (F.col(col) == "NULL") | \
                        (F.lower(F.col(col)) == "other")
        
        # 2. Count for the report
        dirty_count = df.filter(is_dirty_cond).count()
        
        if dirty_count > 0:
            # 3. Apply the replacement
            # "When it's dirty, make it Unknown; otherwise, keep the original column"
            df = df.withColumn(col, 
                               F.when(is_dirty_cond, "Unknown")
                                .otherwise(F.col(col)))
            
            logs = [f"Replaced {dirty_count} rows (null/empty/other) with 'Unknown' in {col}"]
            add_to_report("Value Imputation", col, logs)
            
    return df

columns_to_fill = ["manufacturer", "condition", "fuel", "type", "drive"]
df_categorical_filled = fill_with_unknown(df_no_nulls, columns_to_fill)

In [10]:
def fill_numerical_missing(df, columns):

    for col in columns:
        # Calculate the median (0.5 quantile)
        # approxQuantile returns a list, so we take the first element [0]
        median_val = df.approxQuantile(col, [0.5], 0.01)[0]
        
        df = df.fillna({col: median_val})
        
        logs = [f"Filled missing values with median: {median_val}"]
        add_to_report("Numerical Imputation", col, logs)
    return df

numerical_cols_to_fill = ["cylinders_numeric", "odometer"]
df_numerical_filled = fill_numerical_missing(df_categorical_filled, numerical_cols_to_fill)

In [11]:
def clean_target_column(df, target_col="price"):
    """
    Drops rows where the target is NULL, empty, 
    or logically invalid (<= 0).
    """
    
    invalid_cond = (F.col(target_col).isNull()) | (F.col(target_col) <= 2000)
    
    dropped_count_count = df.filter(invalid_cond).count()
    removed_df = df.filter(invalid_cond)
    removed_df.toPandas().to_csv(f"data/interim/removed_nulls_{target_col}.csv", index=False)
    df_cleaned = df.filter(~invalid_cond)
    logs = [f"Dropped {dropped_count_count} rows with invalid {target_col} (NULL or <= 0)"]
    add_to_report("Target Column Cleaning", target_col, logs)
    
    
    return df_cleaned

# Usage
df_final = clean_target_column(df_numerical_filled, "price")

In [12]:
df_final= df_final[["url", "city", "price", "cylinders_numeric", "odometer", "manufacturer", "condition", "fuel", "type", "title_status", "transmission", "drive", "lat", "long", "year"]]
df_final.toPandas().to_csv("data/interim/used_cars_cleaned.csv", index=False)

In [13]:
size_in_bytes = df_final._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()
afterdata = {
            'total_rows': df_final.count(),
            'total_columns': len(df_final.columns),
            'memory_usage': f"{size_in_bytes / 1024**2:.2f} MB",
            'scan_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

In [14]:
def generate_report(json_path="cleaning_report.json"):
        """Saves cleaning results to JSON and CSV and returns summary statistics."""
        
        report_data = {
            "logs": cleaning_logs,
            "beforedata": beforedata,
            "afterdata": afterdata
        }

        with open(json_path, 'w') as f:
            json.dump(report_data, f, indent=4)


        
        print(f"\n{'='*55}\n Cleaning COMPLETE\n{'='*55}")
        print(f"Files Saved: {json_path}")

        return report_data

generate_report("Reports/Results/cleaning_report.json")


 Cleaning COMPLETE
Files Saved: Reports/Results/cleaning_report.json


{'logs': [{'timestamp': '2026-05-01 20:53:27',
   'check_type': 'Inconsistency Cleaning',
   'column_affected': 'cylinders',
   'log': ['Cylinders column cleaned of inconsistencies.']},
  {'timestamp': '2026-05-01 20:53:28',
   'check_type': 'Data Type Validation',
   'column_affected': 'All',
   'log': ['Columns casted to expected types.']},
  {'timestamp': '2026-05-01 20:53:44',
   'check_type': 'Outlier Capping',
   'column_affected': 'price',
   'log': ['Capped price at upper bound: 39050.00']},
  {'timestamp': '2026-05-01 20:54:01',
   'check_type': 'Outlier Capping',
   'column_affected': 'odometer',
   'log': ['Capped odometer at upper bound: 272410.00']},
  {'timestamp': '2026-05-01 20:54:12',
   'check_type': 'Outlier Capping',
   'column_affected': 'lat',
   'log': ['Capped lat at lower bound: 23.31']},
  {'timestamp': '2026-05-01 20:54:17',
   'check_type': 'Outlier Capping',
   'column_affected': 'lat',
   'log': ['Capped lat at upper bound: 53.83']},
  {'timestamp': '2026-